In [ ]:
import os
import time
import sys
import clr
import numpy as np
import matplotlib.pyplot as plt
from slmcontrol import *
from structured_optics import *
from structured_optics.slmdisplay import My_SLM
from structured_optics.hologram import slm_hologram
import time 
import tqdm
from pylablib.devices import Thorlabs
from slmcontrol import *
from cameras.Ximea import XimeaCamera
from cameras.ImagingSource import ImagingSourceCamera


# Add References to .NET libraries

clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.DeviceManagerCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\Thorlabs.MotionControl.GenericMotorCLI.dll")
clr.AddReference("C:\\Program Files\\Thorlabs\\Kinesis\\ThorLabs.MotionControl.KCube.StepperMotorCLI.dll")
from Thorlabs.MotionControl.DeviceManagerCLI import *
from Thorlabs.MotionControl.GenericMotorCLI import *
from Thorlabs.MotionControl.KCube.StepperMotorCLI import *
from System import Decimal  # necessary for real world units




In [ ]:
#params
z = 20e-2 #Distancia do waist do feixe até a amostra
resolution = 100
step_size = 100e-9
nome_amostra = 'ferro'

In [ ]:
#init cam
camera = ImagingSourceCamera()


#init slm
slm_pixel = 8e-6

slm = My_SLM(screen_index=1)

#init beam
beam = Beam(nix = slm_pixel*slm.width/2, Dx = slm.width,
            niy = slm_pixel*slm.height/2, Dy = slm.height,
            waist = 0.5e-3, lamb = 633e-9)

#init motors and home it
stage_z = Thorlabs.kinesis.KinesisMotor('26005619',scale='ZST',  default_channel=2, is_rack_system=False)
print(f" stage z  -- {stage_z.get_position(scale= 'ZST')}")
stages = [stage_z]



for _ in stages:
    print(_.is_homed())
    if _.is_homed() == False:
        _.home()
for _ in stages:
    _.wait_for_home()



In [ ]:

hg_modes = [[0,0], [0,1], [1,0], [2,0], [1,1], [0,2], [3,0], [2,1], [1,2], [0,3], [4,0], [3,1], [2,2], [1,3], [0,4], [5,0], [4,1], [3,2], [2,3], [1,4], [0,5]]
lg_modes = [[0,1], [0,-1], [0,2], [1,0], [0,-2], [0,3], [1,1], [1,-1], [0,-3], [0,4], [1, 2], [1,-2], [2, 0], [0,-4], [0,5], [1,3], [1, -3], [2, 1], [2,-1], [0,-5]]


for size in np.linspace(0.3, 0.6, 7):
    for mode in range(len(lg_modes)):
        stage_z.move_to(0.003) # in meter
        stage_z.wait_move()
        stage_z.move_to(0.004) # in meter
        stage_z.wait_move()

        data = np.empty((resolution,480, 640),dtype="uint8")

        print('measuring mode lg:', lg_modes[mode], 'size:', size)
        n = lg_modes[mode][0]
        m = lg_modes[mode][1]

        beam.waist = size*1e-3
        beam.lg(l=m, p=n)
        holo = slm_hologram(beam, 3, 4, max_range=200)
        slm.update_image(holo)
        time.sleep(0.1)


        for i in range(resolution):
            stage_z.move_by(step_size)
            stage_z.wait_move()
            time.sleep(0.1)
            data[i] = camera.capture()

        np.savez(f'Amostras_10_2026\\amostra{nome_amostra}_lg{n}{m}_step{int(step_size*1e9)}e-9_Resolution{resolution}_modesize{size:.2f}mm', data)
    
print("lg end")

for size in np.linspace(0.3, 0.6, 7):
    for mode in range(len(hg_modes)):
        stage_z.move_to(0.003) # in meter
        stage_z.wait_move()
        stage_z.move_to(0.004) # in meter
        stage_z.wait_move()

        data = np.empty((resolution,480, 640),dtype="uint8")

        print('measuring mode hg:', hg_modes[mode], 'size:', size)
        n = hg_modes[mode][0]
        m = hg_modes[mode][1]

        beam.waist = size*1e-3
        beam.hg(n=n, m=m)
        holo = slm_hologram(beam, 3, 4, max_range=200)
        slm.update_image(holo)
        time.sleep(0.1)


        for i in range(resolution):
            stage_z.move_by(step_size)
            stage_z.wait_move()
            time.sleep(0.1)
            data[i] = camera.capture()

        np.savez(f'Amostras_10_2026\\amostra{nome_amostra}_hg{n}{m}_step{int(step_size*1e9)}e-9_Resolution{resolution}_modesize{size:.2f}mm', data)

print('hg end')

In [ ]:
#For fixed std

#hg_modes = [[0,0], [0,1], [1,0], [2,0], [1,1], [0,2], [3,0], [2,1], [1,2], [0,3], [4,0], [3,1], [2,2], [1,3], [0,4], [5,0], [4,1], [3,2], [2,3], [1,4], [0,5]]
#lg_modes = [[0,1], [0,-1], [0,2], [1,0], [0,-2], [0,3], [1,1], [1,-1], [0,-3], [0,4], [1, 2], [1,-2], [2, 0], [0,-4], [0,5], [1,3], [1, -3], [2, 1], [2,-1], [0,-5]]

C = 0.25e-3

for mode in range(len(lg_modes)):
    stage_z.move_to(0.003) # in meter
    stage_z.wait_move()
    stage_z.move_to(0.004) # in meter
    stage_z.wait_move()

    data = np.empty((resolution,480, 640),dtype="uint8")

    print('measuring fixed std lg:', lg_modes[mode])
    n = lg_modes[mode][0]
    m = lg_modes[mode][1]

    beam.waist = np.sqrt(2*C**2/(2*n + np.abs(m) +1) + 2*np.sqrt(C**4/(2*n + np.abs(m) +1)**2 - z**2/beam.k()**2))
    beam.waist = size*1e-3
    beam.lg(l=m, p=n)
    holo = slm_hologram(beam, 3, 4, max_range=200)
    slm.update_image(holo)
    time.sleep(0.1)


    for i in range(resolution):
        stage_z.move_by(step_size)
        stage_z.wait_move()
        time.sleep(0.1)
        data[i] = camera.capture()

    np.savez(f'Amostras_10_2026\\amostra{nome_amostra}_lg{n}{m}_step{int(step_size*1e9)}e-9_Resolution{resolution}_fixedstd', data)


for mode in range(len(hg_modes)):
    stage_z.move_to(0.003) # in meter
    stage_z.wait_move()
    stage_z.move_to(0.004) # in meter
    stage_z.wait_move()

    data = np.empty((resolution,480, 640),dtype="uint8")

    print('measuring mode hg:', hg_modes[mode])
    n = hg_modes[mode][0]
    m = hg_modes[mode][1]

    beam.waist = np.sqrt(2*C**2/(2*n+1) + 2*np.sqrt(C**4/(2*n+1)**2 - z**2/beam.k()**2))
    beam.hg(n=n, m=0)
    holo = slm_hologram(beam, 3, 4, max_range=200)
    slm.update_image(holo)
    time.sleep(0.1)


    for i in range(resolution):
        stage_z.move_by(step_size)
        stage_z.wait_move()
        time.sleep(0.1)
        data[i] = camera.capture()

    np.savez(f'Amostras_10_2026\\amostra{nome_amostra}_hg{n}{m}_step{int(step_size*1e9)}e-9_Resolution{resolution}_fixedstd', data)
